In [43]:
import os
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()
client = Anthropic()
print("Client created successfully")


Client created successfully


In [44]:
response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=500,
    system="You are an AML triage analyst. Be concise.",
    messages=[
        {
            "role": "user",
            "content": "What are the three most common AML red flags in cross-border wire transfers? Answer in one short paragraph."
        }
    ]
)
print(response.content[0].text)

The three most common AML red flags in cross-border wire transfers are: (1) **Unusual beneficiary characteristics** — transfers to high-risk jurisdictions, politically exposed persons (PEPs), or unrelated third parties rather than stated business partners; (2) **Inconsistent transaction patterns** — sudden spikes in transfer frequency or volume, round-dollar amounts, or transfers that contradict the customer's business profile; and (3) **Structural indicators** — rapid movement of funds through multiple intermediaries, use of shell companies, lack of clear business purpose, or transfers involving sanctioned countries or entities. These often indicate potential money laundering, sanctions evasion, or terrorist financing concerns.


In [45]:
print(f"Input tokens: {response.usage.input_tokens}")
print(f"Output tokens: {response.usage.output_tokens}")


Input tokens: 43
Output tokens: 150


In [46]:
case_1 = {
    "case_id": "C001",
    "entity_name": "Northwind Trading LLC",
    "entity_type": "Import/Export Business",
    "jurisdiction": "Cyprus",
    "beneficial_owner": "Unknown - nominee director listed",
    "transaction_pattern": "Received 6 incoming wires of $9,800 each over 8 days from different senders in UAE, immediately forwarded 95% to an account in Latvia within 24 hours each time.",
    "stated_business_purpose": "Import of electronics components",
    "recent_news": "No adverse media found.",
}

In [47]:
# Target output shape for every case:
# {
#   "case_id": str,
#   "risk_score": int (0-100),
#   "risk_category": str ("low" | "medium" | "high"),
#   "red_flags": list of strings,
#   "reasoning": str (2-3 sentences),
#   "requires_human_review": bool
# }

In [48]:
SYSTEM_PROMPT = """You are an AML (Anti-Money Laundering) triage analyst. You evaluate counterparty profiles for money laundering risk based on FATF red flag typologies, including structuring, layering, unusual beneficial ownership, and business-purpose mismatches.

For every case you are given, respond with ONLY a valid JSON object in exactly this shape, with no other text before or after it:

{
  "case_id": "<the case id from the input>",
  "risk_score": <integer 0-100>,
  "risk_category": "<low, medium, or high>",
  "red_flags": ["<flag 1>", "<flag 2>", ...],
  "reasoning": "<2-3 sentence explanation>",
  "requires_human_review": <true or false>
}

Do not include any text outside the JSON object. Do not use markdown formatting or code blocks. Return raw JSON only."""

In [49]:
import json

def triage_case(case, model="claude-haiku-4-5"):
    case_text = json.dumps(case, indent=2)
    
    response = client.messages.create(
        model=model,
        max_tokens=500,
        system=SYSTEM_PROMPT,
        messages=[
            {"role": "user", "content": f"Evaluate this case:\n\n{case_text}"}
        ]
    )
    
    raw_text = response.content[0].text
    cleaned_text = extract_json(raw_text)
    result = json.loads(cleaned_text)
    return result, response.usage

In [78]:
def extract_json(raw_text):
    """
    Defensively cleans Claude's response to extract pure JSON.
    Handles markdown fences, prose introductions, and whitespace.
    """
    if not raw_text or not raw_text.strip():
        raise ValueError("Empty response from Claude")
    
    text = raw_text.strip()
    
    # Strip markdown code fences if present
    if text.startswith("```"):
        lines = text.split("\n")
        text = "\n".join(lines[1:-1]).strip()
    
    # Handle prose introduction before JSON
    # Find the first { and last } and extract just that
    start = text.find("{")
    end = text.rfind("}")
    
    if start == -1 or end == -1:
        raise ValueError(f"No JSON object found in response: {repr(text[:100])}")
    
    text = text[start:end+1]
    
    if not text:
        raise ValueError("Empty after JSON extraction")
    
    return text

print("extract_json updated - handles prose introductions")

extract_json updated - handles prose introductions


In [51]:
result, usage = triage_case(case_1)
print(json.dumps(result, indent=2))
print(f"\nTokens used: {usage.input_tokens} in, {usage.output_tokens} out")

{
  "case_id": "C001",
  "risk_score": 92,
  "risk_category": "high",
  "red_flags": [
    "Structuring: Six identical wire amounts of $9,800 (just below $10,000 CTR threshold) over short timeframe",
    "Layering: Immediate 95% forwarding to third-country (Latvia) within 24 hours indicates pass-through activity",
    "Beneficial ownership opacity: Unknown beneficial owner with nominee director arrangement",
    "High-risk jurisdictions: Cyprus and Latvia combination typical of layering schemes",
    "Business-purpose mismatch: Import/export business stated but transaction pattern shows no commercial substance",
    "Multiple senders from UAE: Diffuse source of funds suggests orchestrated structuring scheme"
  ],
  "reasoning": "This entity exhibits classic money laundering indicators including structuring below reporting thresholds, rapid layering through high-risk jurisdictions, obscured beneficial ownership, and transaction patterns inconsistent with legitimate import/export operati

In [52]:
case_text = json.dumps(case_1, indent=2)

response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=500,
    system=SYSTEM_PROMPT,
    messages=[
        {"role": "user", "content": f"Evaluate this case:\n\n{case_text}"}
    ]
)

raw_text = response.content[0].text
print(repr(raw_text))

'```json\n{\n  "case_id": "C001",\n  "risk_score": 85,\n  "risk_category": "high",\n  "red_flags": [\n    "Structuring - Multiple transactions just below $10,000 threshold across 8 days",\n    "Unknown beneficial ownership - Nominee director arrangement",\n    "Rapid layering - 95% of funds forwarded to Latvia within 24 hours",\n    "Multiple foreign sources - Six different UAE senders",\n    "Jurisdiction risk - Cyprus combined with Latvia",\n    "Business-purpose mismatch - Import/export business pattern inconsistent with rapid fund transfers"\n  ],\n  "reasoning": "This case exhibits classic structuring, layering, and beneficial ownership red flags consistent with money laundering typologies. The precise $9,800 transactions designed to evade reporting thresholds, combined with rapid offshore transfers and opaque ownership structure, create significant concern. The stated import/export business purpose does not align with the immediate fund forwarding behavior.",\n  "requires_human_r

In [53]:
def extract_json(raw_text):
    text = raw_text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1]
        text = text.rsplit("```", 1)[0]
    return text.strip()

In [54]:
case_2 = {
    "case_id": "C002",
    "entity_name": "Riverside Family Dental Practice",
    "entity_type": "Healthcare Provider",
    "jurisdiction": "Ireland",
    "beneficial_owner": "Dr. Mairead Collins, sole practitioner, verified",
    "transaction_pattern": "Monthly incoming payments from Irish health insurers (VHI, Laya) ranging €8,000-€15,000, consistent with patient billing cycles. Quarterly equipment supplier payments to known dental equipment manufacturers in Germany.",
    "stated_business_purpose": "Dental practice operations",
    "recent_news": "No adverse media found.",
}

case_3 = {
    "case_id": "C003",
    "entity_name": "Apex Consulting Partners",
    "entity_type": "Management Consulting",
    "jurisdiction": "United Arab Emirates",
    "beneficial_owner": "Listed as a holding company in British Virgin Islands; ultimate individual owner not disclosed",
    "transaction_pattern": "Receives quarterly 'advisory fee' payments of $40,000-$60,000 from three different mid-sized manufacturing firms in Eastern Europe. No employees on LinkedIn, no verifiable client testimonials or public engagements.",
    "stated_business_purpose": "Strategic management consulting services",
    "recent_news": "One of the three paying manufacturing firms was named in a 2025 local news article regarding a public procurement irregularity, unrelated to this entity directly.",
}

In [55]:
for case in [case_2, case_3]:
    result, usage = triage_case(case)
    print(json.dumps(result, indent=2))
    print(f"Tokens: {usage.input_tokens} in, {usage.output_tokens} out")
    print("-" * 60)
    

{
  "case_id": "C002",
  "risk_score": 12,
  "risk_category": "low",
  "red_flags": [],
  "reasoning": "Riverside Family Dental Practice exhibits a straightforward business model with transparent beneficial ownership, legitimate income sources from regulated Irish health insurers, and expense patterns consistent with dental operations. Transaction amounts and frequencies align with typical healthcare provider billing cycles, and the verified sole practitioner ownership structure eliminates layering concerns.",
  "requires_human_review": false
}
Tokens: 383 in, 127 out
------------------------------------------------------------
{
  "case_id": "C003",
  "risk_score": 72,
  "risk_category": "high",
  "red_flags": [
    "Beneficial ownership obscured through BVI holding company structure with non-disclosed ultimate individual owner",
    "No verifiable business presence: zero LinkedIn employees, no client testimonials, no public engagement records",
    "Business-purpose mismatch: stated 

In [56]:
pip install requests


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [57]:
import requests

def check_sanctions(entity_name):
    """
    Queries OpenSanctions API for a given entity name.
    Returns a summary of whether the entity appears on any sanctions list.
    """
    try:
        url = "https://api.opensanctions.org/match/default"
        payload = {
            "queries": {
                "q1": {
                    "schema": "LegalEntity",
                    "properties": {
                        "name": [entity_name]
                    }
                }
            }
        }
        response = requests.post(url, json=payload, timeout=10)
        data = response.json()
        
        results = data.get("responses", {}).get("q1", {}).get("results", [])
        
        if not results:
            return {"sanctioned": False, "matches": [], "summary": f"No sanctions matches found for '{entity_name}'."}
        
        matches = []
        for r in results[:3]:
            matches.append({
                "name": r.get("caption", "Unknown"),
                "score": r.get("score", 0),
                "datasets": r.get("datasets", [])
            })
        
        return {
            "sanctioned": True,
            "matches": matches,
            "summary": f"Found {len(results)} potential sanctions match(es) for '{entity_name}'."
        }
    
    except Exception as e:
        return {"sanctioned": False, "matches": [], "summary": f"Sanctions check unavailable: {str(e)}"}

In [58]:
test_result = check_sanctions("Viktor Bout")
print(json.dumps(test_result, indent=2))

{
  "sanctioned": false,
  "matches": [],
  "summary": "No sanctions matches found for 'Viktor Bout'."
}


In [59]:
FATF_RISK_RATINGS = {
    "Iran": "blacklist",
    "North Korea": "blacklist", 
    "Myanmar": "blacklist",
    "Russia": "greylist",
    "UAE": "greylist",
    "Cyprus": "greylist",
    "Malta": "greylist",
    "British Virgin Islands": "high_risk",
    "Cayman Islands": "high_risk",
    "Panama": "high_risk",
    "Latvia": "elevated",
    "Estonia": "elevated",
    "Ireland": "standard",
    "Germany": "standard",
    "United Kingdom": "standard",
    "United States": "standard",
    "France": "standard",
    "Netherlands": "standard",
    "United Arab Emirates": "greylist",
}

def get_jurisdiction_risk(country):
    """
    Returns FATF risk rating for a given country.
    """
    rating = FATF_RISK_RATINGS.get(country, "unknown")
    
    explanations = {
        "blacklist": "FATF blacklist — subject to enhanced due diligence and potential transaction restrictions.",
        "greylist": "FATF greylist — under increased monitoring for AML/CFT deficiencies.",
        "high_risk": "High-risk offshore jurisdiction — commonly associated with secrecy and limited transparency.",
        "elevated": "Elevated risk — historical AML concerns, increased scrutiny recommended.",
        "standard": "Standard risk — no significant FATF concerns currently noted.",
        "unknown": "Not in database — manual assessment recommended."
    }
    
    return {
        "country": country,
        "fatf_rating": rating,
        "explanation": explanations[rating]
    }

In [60]:
print(json.dumps(get_jurisdiction_risk("Cyprus"), indent=2))
print(json.dumps(get_jurisdiction_risk("Ireland"), indent=2))

{
  "country": "Cyprus",
  "fatf_rating": "greylist",
  "explanation": "FATF greylist \u2014 under increased monitoring for AML/CFT deficiencies."
}
{
  "country": "Ireland",
  "fatf_rating": "standard",
  "explanation": "Standard risk \u2014 no significant FATF concerns currently noted."
}


In [61]:
tools = [
    {
        "name": "check_sanctions",
        "description": "Checks whether an entity name appears on international sanctions lists via OpenSanctions database. Use this for any entity whose sanctions status is unclear or unverified.",
        "input_schema": {
            "type": "object",
            "properties": {
                "entity_name": {
                    "type": "string",
                    "description": "The full legal name of the entity or individual to check"
                }
            },
            "required": ["entity_name"]
        }
    },
    {
        "name": "get_jurisdiction_risk",
        "description": "Returns FATF risk rating for a given country or jurisdiction. Use this to assess the risk level of any jurisdiction mentioned in a case.",
        "input_schema": {
            "type": "object",
            "properties": {
                "country": {
                    "type": "string",
                    "description": "The country or jurisdiction name to look up"
                }
            },
            "required": ["country"]
        }
    }
]

In [75]:
def triage_case_with_tools(case, model="claude-haiku-4-5"):
    case_text = json.dumps(case, indent=2)
    
    # Start the conversation with the case
    messages = [
        {
            "role": "user",
            "content": f"Evaluate this AML case using the available tools:\n\n{case_text}"
        }
    ]
    
    tool_calls_log = []
    
    # The agentic loop — keeps running until Claude gives a final answer
    while True:
        response = client.messages.create(
            model=model,
            max_tokens=1000,
            system=SYSTEM_PROMPT,
            tools=tools,
            messages=messages
        )
        
        # Case 1: Claude wants to call a tool
        if response.stop_reason == "tool_use":
            
            # Add Claude's response to conversation history
            messages.append({
                "role": "assistant",
                "content": response.content
            })
            
            # Find which tool Claude wants to call
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    tool_name = block.name
                    tool_input = block.input
                    
                    print(f"Claude is calling: {tool_name}({tool_input})")
                    
                    # Actually run the tool
                    if tool_name == "check_sanctions":
                        result = check_sanctions(tool_input["entity_name"])
                    elif tool_name == "get_jurisdiction_risk":
                        result = get_jurisdiction_risk(tool_input["country"])
                    else:
                        result = {"error": f"Unknown tool: {tool_name}"}
                    
                    tool_calls_log.append({
                        "tool": tool_name,
                        "input": tool_input,
                        "output": result
                    })
                    
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": json.dumps(result)
                    })
            
            # Send tool results back to Claude
            messages.append({
                "role": "user",
                "content": tool_results
            })
        
        # Case 2: Claude has finished reasoning — final answer
        elif response.stop_reason == "end_turn":
            raw_text = ""
            for block in response.content:
                if hasattr(block, 'text') and block.text.strip():
                    raw_text = block.text
                    break
            
            if not raw_text:
                raise ValueError("No text content in final response")
            
            cleaned_text = extract_json(raw_text)
            final_result = json.loads(cleaned_text)
            return final_result, response.usage, tool_calls_log

In [63]:
result, usage, tool_log = triage_case_with_tools(case_1)

print("\n=== TOOL CALLS MADE ===")
for call in tool_log:
    print(f"\nTool: {call['tool']}")
    print(f"Input: {call['input']}")
    print(f"Output: {json.dumps(call['output'], indent=2)}")

print("\n=== FINAL TRIAGE RESULT ===")
print(json.dumps(result, indent=2))
print(f"\nTokens used: {usage.input_tokens} in, {usage.output_tokens} out")

Claude is calling: check_sanctions({'entity_name': 'Northwind Trading LLC'})
Claude is calling: get_jurisdiction_risk({'country': 'Cyprus'})
Claude is calling: get_jurisdiction_risk({'country': 'Latvia'})
Claude is calling: get_jurisdiction_risk({'country': 'United Arab Emirates'})

=== TOOL CALLS MADE ===

Tool: check_sanctions
Input: {'entity_name': 'Northwind Trading LLC'}
Output: {
  "sanctioned": false,
  "matches": [],
  "summary": "No sanctions matches found for 'Northwind Trading LLC'."
}

Tool: get_jurisdiction_risk
Input: {'country': 'Cyprus'}
Output: {
  "country": "Cyprus",
  "fatf_rating": "greylist",
  "explanation": "FATF greylist \u2014 under increased monitoring for AML/CFT deficiencies."
}

Tool: get_jurisdiction_risk
Input: {'country': 'Latvia'}
Output: {
  "country": "Latvia",
  "fatf_rating": "elevated",
  "explanation": "Elevated risk \u2014 historical AML concerns, increased scrutiny recommended."
}

Tool: get_jurisdiction_risk
Input: {'country': 'United Arab Emi

In [64]:
result, usage, tool_log = triage_case_with_tools(case_2)

print("\n=== TOOL CALLS MADE ===")
for call in tool_log:
    print(f"\nTool: {call['tool']}")
    print(f"Input: {call['input']}")
    print(f"Output: {json.dumps(call['output'], indent=2)}")

print("\n=== FINAL TRIAGE RESULT ===")
print(json.dumps(result, indent=2))
print(f"\nTokens used: {usage.input_tokens} in, {usage.output_tokens} out")

Claude is calling: check_sanctions({'entity_name': 'Riverside Family Dental Practice'})
Claude is calling: check_sanctions({'entity_name': 'Dr. Mairead Collins'})
Claude is calling: get_jurisdiction_risk({'country': 'Ireland'})

=== TOOL CALLS MADE ===

Tool: check_sanctions
Input: {'entity_name': 'Riverside Family Dental Practice'}
Output: {
  "sanctioned": false,
  "matches": [],
  "summary": "No sanctions matches found for 'Riverside Family Dental Practice'."
}

Tool: check_sanctions
Input: {'entity_name': 'Dr. Mairead Collins'}
Output: {
  "sanctioned": false,
  "matches": [],
  "summary": "No sanctions matches found for 'Dr. Mairead Collins'."
}

Tool: get_jurisdiction_risk
Input: {'country': 'Ireland'}
Output: {
  "country": "Ireland",
  "fatf_rating": "standard",
  "explanation": "Standard risk \u2014 no significant FATF concerns currently noted."
}

=== FINAL TRIAGE RESULT ===
{
  "case_id": "C002",
  "risk_score": 8,
  "risk_category": "low",
  "red_flags": [],
  "reasoning": 

In [65]:
result, usage, tool_log = triage_case_with_tools(case_3)
                                                
print("\n=== TOOL CALLS MADE ===")
for call in tool_log:
    print(f"\nTool: {call['tool']}")
    print(f"Input: {call['input']}")
    print(f"Output: {json.dumps(call['output'], indent=2)}")

print("\n=== FINAL TRIAGE RESULT ===")
print(json.dumps(result, indent=2))
print(f"\nTokens used: {usage.input_tokens} in, {usage.output_tokens} out")

Claude is calling: check_sanctions({'entity_name': 'Apex Consulting Partners'})
Claude is calling: get_jurisdiction_risk({'country': 'United Arab Emirates'})
Claude is calling: get_jurisdiction_risk({'country': 'British Virgin Islands'})
Claude is calling: get_jurisdiction_risk({'country': 'Eastern Europe'})

=== TOOL CALLS MADE ===

Tool: check_sanctions
Input: {'entity_name': 'Apex Consulting Partners'}
Output: {
  "sanctioned": false,
  "matches": [],
  "summary": "No sanctions matches found for 'Apex Consulting Partners'."
}

Tool: get_jurisdiction_risk
Input: {'country': 'United Arab Emirates'}
Output: {
  "country": "United Arab Emirates",
  "fatf_rating": "greylist",
  "explanation": "FATF greylist \u2014 under increased monitoring for AML/CFT deficiencies."
}

Tool: get_jurisdiction_risk
Input: {'country': 'British Virgin Islands'}
Output: {
  "country": "British Virgin Islands",
  "fatf_rating": "high_risk",
  "explanation": "High-risk offshore jurisdiction \u2014 commonly ass

In [66]:
test_cases = [
    # CATEGORY 1: OBVIOUS HIGH RISK (5 cases)
    {
        "case_id": "C001",
        "entity_name": "Northwind Trading LLC",
        "entity_type": "Import/Export Business",
        "jurisdiction": "Cyprus",
        "beneficial_owner": "Unknown - nominee director listed",
        "transaction_pattern": "Received 6 incoming wires of $9,800 each over 8 days from different senders in UAE, immediately forwarded 95% to Latvia within 24 hours each time.",
        "stated_business_purpose": "Import of electronics components",
        "recent_news": "No adverse media found.",
        "ground_truth": "high"
    },
    {
        "case_id": "C004",
        "entity_name": "Rashid Al-Farsi",
        "entity_type": "Individual",
        "jurisdiction": "Iran",
        "beneficial_owner": "Self",
        "transaction_pattern": "Multiple cash deposits of $4,500 across 5 different bank branches in one week, then single wire transfer of $22,000 to a cryptocurrency exchange.",
        "stated_business_purpose": "Personal savings transfer",
        "recent_news": "No adverse media found.",
        "ground_truth": "high"
    },
    {
        "case_id": "C005",
        "entity_name": "Pacific Rim Holdings",
        "entity_type": "Investment Vehicle",
        "jurisdiction": "Cayman Islands",
        "beneficial_owner": "Three layered holding companies, ultimate beneficial owner not identified",
        "transaction_pattern": "Receives monthly transfers of $200,000-$250,000 from shell companies in Panama and BVI, distributes to 12 different individual accounts across 8 countries within 48 hours.",
        "stated_business_purpose": "Investment fund distributions",
        "recent_news": "No adverse media found.",
        "ground_truth": "high"
    },
    {
        "case_id": "C006",
        "entity_name": "Golden Star Import Export",
        "entity_type": "Trade Finance",
        "jurisdiction": "Myanmar",
        "beneficial_owner": "Listed as government-affiliated entity",
        "transaction_pattern": "Large irregular transfers ranging $50,000-$500,000 with no consistent pattern, counterparties change with each transaction.",
        "stated_business_purpose": "General trade",
        "recent_news": "Parent ministry named in UN sanctions report 2024.",
        "ground_truth": "high"
    },
    {
        "case_id": "C007",
        "entity_name": "Senator Ricardo Mendez Family Trust",
        "entity_type": "Trust",
        "jurisdiction": "Panama",
        "beneficial_owner": "Ricardo Mendez, active government official",
        "transaction_pattern": "Quarterly transfers of $75,000 from anonymous offshore accounts to real estate purchases in Miami and London.",
        "stated_business_purpose": "Family wealth management",
        "recent_news": "Senator Mendez under investigation for public procurement irregularities per local news 2025.",
        "ground_truth": "high"
    },

    # CATEGORY 2: OBVIOUS LOW RISK (5 cases)
    {
        "case_id": "C002",
        "entity_name": "Riverside Family Dental Practice",
        "entity_type": "Healthcare Provider",
        "jurisdiction": "Ireland",
        "beneficial_owner": "Dr. Mairead Collins, sole practitioner, verified",
        "transaction_pattern": "Monthly incoming payments from Irish health insurers VHI and Laya ranging €8,000-€15,000, quarterly equipment payments to German dental suppliers.",
        "stated_business_purpose": "Dental practice operations",
        "recent_news": "No adverse media found.",
        "ground_truth": "low"
    },
    {
        "case_id": "C008",
        "entity_name": "Mueller & Schmidt GmbH",
        "entity_type": "Manufacturing",
        "jurisdiction": "Germany",
        "beneficial_owner": "Hans Mueller and Peter Schmidt, verified directors, publicly listed",
        "transaction_pattern": "Regular monthly payroll transfers to 45 employees, quarterly supplier payments to verified EU manufacturers, annual dividend payments to two identified shareholders.",
        "stated_business_purpose": "Industrial components manufacturing",
        "recent_news": "Won regional business excellence award 2024.",
        "ground_truth": "low"
    },
    {
        "case_id": "C009",
        "entity_name": "Trinity College Dublin Research Fund",
        "entity_type": "Educational Institution",
        "jurisdiction": "Ireland",
        "beneficial_owner": "Trinity College Dublin, public university, fully regulated",
        "transaction_pattern": "Receives annual EU Horizon research grants of €500,000-€2M, disburses to verified academic suppliers and researcher salaries on documented project timelines.",
        "stated_business_purpose": "Academic research funding",
        "recent_news": "Published three peer-reviewed papers funded by this grant in 2025.",
        "ground_truth": "low"
    },
    {
        "case_id": "C010",
        "entity_name": "Brennan Family Solicitors",
        "entity_type": "Legal Services",
        "jurisdiction": "Ireland",
        "beneficial_owner": "Siobhan Brennan, Law Society of Ireland registered",
        "transaction_pattern": "Client account transactions consistent with property conveyancing — receives funds matching purchase prices, disburses to Land Registry and identified counterparties within standard legal timeframes.",
        "stated_business_purpose": "Property conveyancing services",
        "recent_news": "No adverse media found.",
        "ground_truth": "low"
    },
    {
        "case_id": "C011",
        "entity_name": "Dublin Bus Pension Fund",
        "entity_type": "Pension Fund",
        "jurisdiction": "Ireland",
        "beneficial_owner": "Board of trustees, all publicly named and regulated by Pensions Authority Ireland",
        "transaction_pattern": "Monthly contributions from Dublin Bus payroll, quarterly investment purchases through regulated Irish brokers, monthly pension disbursements to 2,400 verified retirees.",
        "stated_business_purpose": "Employee pension management",
        "recent_news": "Awarded best-governed pension fund by Irish Pensions Authority 2024.",
        "ground_truth": "low"
    },

    # CATEGORY 3: AMBIGUOUS/MEDIUM RISK (4 cases)
    {
        "case_id": "C003",
        "entity_name": "Apex Consulting Partners",
        "entity_type": "Management Consulting",
        "jurisdiction": "United Arab Emirates",
        "beneficial_owner": "Listed as a holding company in British Virgin Islands, ultimate individual owner not disclosed",
        "transaction_pattern": "Receives quarterly advisory fee payments of $40,000-$60,000 from three different mid-sized manufacturing firms in Eastern Europe. No employees on LinkedIn, no verifiable client testimonials.",
        "stated_business_purpose": "Strategic management consulting services",
        "recent_news": "One paying firm named in 2025 procurement irregularity article.",
        "ground_truth": "medium"
    },
    {
        "case_id": "C012",
        "entity_name": "Sunrise Real Estate Investments",
        "entity_type": "Property Investment",
        "jurisdiction": "Malta",
        "beneficial_owner": "Two EU citizens, passports verified, no PEP status",
        "transaction_pattern": "Purchases residential properties every 3-4 months ranging €200,000-€400,000 using mixed funding — 40% mortgage, 60% cash from an offshore account in Malta.",
        "stated_business_purpose": "Buy-to-let property portfolio",
        "recent_news": "No adverse media found.",
        "ground_truth": "medium"
    },
    {
        "case_id": "C013",
        "entity_name": "Crypto Valley Trading Ltd",
        "entity_type": "Cryptocurrency Trading",
        "jurisdiction": "Estonia",
        "beneficial_owner": "Two verified directors, Estonian residents, no PEP status",
        "transaction_pattern": "High volume of small transactions $500-$2,000 from hundreds of different individual customers globally, aggregated and sent weekly to three major regulated crypto exchanges.",
        "stated_business_purpose": "Retail cryptocurrency brokerage",
        "recent_news": "Licensed by Estonian Financial Intelligence Unit.",
        "ground_truth": "medium"
    },
    {
        "case_id": "C014",
        "entity_name": "Horizon Freight Solutions",
        "entity_type": "Logistics",
        "jurisdiction": "United Arab Emirates",
        "beneficial_owner": "Omar Hassan, UAE national, verified",
        "transaction_pattern": "Receives payments from clients in Russia and Iran for freight forwarding services, pays verified shipping companies in Netherlands and Germany.",
        "stated_business_purpose": "International freight forwarding",
        "recent_news": "No adverse media found.",
        "ground_truth": "medium"
    },

    # CATEGORY 4: ADVERSARIAL CASES (3 cases)
    {
        "case_id": "C015",
        "entity_name": "Clean Hands Charity Foundation",
        "entity_type": "Non-Profit Organisation",
        "jurisdiction": "United Kingdom",
        "beneficial_owner": "Board of trustees, all named and UK Charity Commission registered",
        "transaction_pattern": "Receives public donations averaging £50,000 monthly, sends humanitarian aid payments to partners in Syria and Afghanistan via approved wire transfers.",
        "stated_business_purpose": "Humanitarian aid delivery",
        "recent_news": "No adverse media found.",
        "ground_truth": "medium"
    },
    {
        "case_id": "C016",
        "entity_name": "Premier League Football Club Academy Fund",
        "entity_type": "Sports Organisation",
        "jurisdiction": "United Kingdom",
        "beneficial_owner": "Club ownership group, partially held by sovereign wealth fund from Qatar",
        "transaction_pattern": "Receives large irregular transfers from parent club ranging £500,000-£2M, disburses to youth player agents and international transfer fees.",
        "stated_business_purpose": "Youth academy development",
        "recent_news": "Parent club under UEFA Financial Fair Play investigation 2025.",
        "ground_truth": "medium"
    },
    {
        "case_id": "C017",
        "entity_name": "Wellness & Vitality Supplements Ltd",
        "entity_type": "Retail",
        "jurisdiction": "Ireland",
        "beneficial_owner": "Sarah O'Brien, verified Irish national",
        "transaction_pattern": "Monthly revenue of €15,000-€25,000 from online sales, but 70% of payments received in cryptocurrency converted to EUR through three different exchanges before depositing to business account.",
        "stated_business_purpose": "Online health supplements retail",
        "recent_news": "No adverse media found.",
        "ground_truth": "medium"
    },

    # CATEGORY 5: EDGE CASES (3 cases)
    {
        "case_id": "C018",
        "entity_name": "Babatunde Okonkwo",
        "entity_type": "Individual",
        "jurisdiction": "Nigeria",
        "beneficial_owner": "Self",
        "transaction_pattern": "Receives monthly remittance transfers of $800-$1,200 from family members in the United States and United Kingdom, consistent amounts over 3 years.",
        "stated_business_purpose": "Family remittances",
        "recent_news": "No adverse media found.",
        "ground_truth": "low"
    },
    {
        "case_id": "C019",
        "entity_name": "Northern Lights Cannabis Corp",
        "entity_type": "Cannabis Retail",
        "jurisdiction": "Netherlands",
        "beneficial_owner": "Three verified Dutch nationals, licensed by Dutch authorities",
        "transaction_pattern": "High cash deposits weekly ranging €20,000-€40,000, limited banking relationships due to industry restrictions, uses three different banks to manage cash flow.",
        "stated_business_purpose": "Licensed cannabis retail operations",
        "recent_news": "No adverse media found.",
        "ground_truth": "medium"
    },
    {
        "case_id": "C020",
        "entity_name": "Archbishop Donation Fund",
        "entity_type": "Religious Organisation",
        "jurisdiction": "Vatican City",
        "beneficial_owner": "Holy See, sovereign entity",
        "transaction_pattern": "Receives global donations ranging €10,000-€500,000 from individuals and organisations across 40 countries, disburses to Catholic charitable organisations and church maintenance globally.",
        "stated_business_purpose": "Religious charitable operations",
        "recent_news": "Vatican financial reforms praised by FATF 2024.",
        "ground_truth": "low"
    }
]

print(f"Dataset loaded: {len(test_cases)} cases")

Dataset loaded: 20 cases


In [67]:
def categories_match(predicted, ground_truth):
    """
    Returns True if predicted category matches ground truth.
    Both should be 'low', 'medium', or 'high'.
    """
    return predicted.lower().strip() == ground_truth.lower().strip()

def is_high_risk(category):
    """Returns True if category is high risk."""
    return category.lower().strip() == "high"

print("Helper functions defined")

Helper functions defined


In [68]:
import time
import json
from datetime import datetime

def run_eval(test_cases, model="claude-haiku-4-5", use_tools=True):
    """
    Runs all test cases through the agent and compares against ground truth.
    Returns a detailed results list and summary statistics.
    """
    results = []
    errors = []
    
    print(f"Starting eval: {len(test_cases)} cases, model={model}")
    print(f"Use tools: {use_tools}")
    print("-" * 50)
    
    for i, case in enumerate(test_cases):
        case_id = case["case_id"]
        ground_truth = case["ground_truth"]
        
        print(f"Running case {i+1}/{len(test_cases)}: {case_id}...", end=" ")
        
        try:
            start_time = time.time()
            
            # Run the case through the right function
            if use_tools:
                result, usage, tool_log = triage_case_with_tools(case, model=model)
            else:
                result, usage = triage_case(case, model=model)
                tool_log = []
            
            elapsed = round(time.time() - start_time, 2)
            
            # Compare prediction against ground truth
            predicted_category = result.get("risk_category", "unknown")
            correct = categories_match(predicted_category, ground_truth)
            
            print(f"{'CORRECT' if correct else 'WRONG'} "
                  f"(predicted={predicted_category}, "
                  f"truth={ground_truth}, "
                  f"score={result.get('risk_score')}, "
                  f"time={elapsed}s)")
            
            results.append({
                "case_id": case_id,
                "ground_truth": ground_truth,
                "predicted_category": predicted_category,
                "predicted_score": result.get("risk_score"),
                "correct": correct,
                "red_flags": result.get("red_flags", []),
                "reasoning": result.get("reasoning", ""),
                "requires_human_review": result.get("requires_human_review"),
                "tool_calls_count": len(tool_log),
                "input_tokens": usage.input_tokens,
                "output_tokens": usage.output_tokens,
                "latency_seconds": elapsed,
            })
            
            # Small delay to avoid rate limiting
            time.sleep(1)
            
        except Exception as e:
            print(f"ERROR: {str(e)}")
            errors.append({"case_id": case_id, "error": str(e)})
    
    print("-" * 50)
    print(f"Eval complete. {len(results)} succeeded, {len(errors)} errors.")
    
    return results, errors

print("Eval runner defined")

Eval runner defined


In [69]:
def calculate_metrics(results):
    """
    Calculates precision, recall, F1, and accuracy from eval results.
    """
    total = len(results)
    correct = sum(1 for r in results if r["correct"])
    accuracy = round(correct / total * 100, 1)
    
    # Binary classification: high risk vs not high risk
    tp = sum(1 for r in results 
             if is_high_risk(r["predicted_category"]) 
             and is_high_risk(r["ground_truth"]))
    
    fp = sum(1 for r in results 
             if is_high_risk(r["predicted_category"]) 
             and not is_high_risk(r["ground_truth"]))
    
    fn = sum(1 for r in results 
             if not is_high_risk(r["predicted_category"]) 
             and is_high_risk(r["ground_truth"]))
    
    tn = sum(1 for r in results 
             if not is_high_risk(r["predicted_category"]) 
             and not is_high_risk(r["ground_truth"]))
    
    precision = round(tp / (tp + fp) * 100, 1) if (tp + fp) > 0 else 0
    recall = round(tp / (tp + fn) * 100, 1) if (tp + fn) > 0 else 0
    f1 = round(2 * precision * recall / (precision + recall), 1) if (precision + recall) > 0 else 0
    
    avg_tokens = round(sum(r["input_tokens"] for r in results) / total)
    avg_latency = round(sum(r["latency_seconds"] for r in results) / total, 2)
    avg_tool_calls = round(sum(r["tool_calls_count"] for r in results) / total, 1)
    
    return {
        "total_cases": total,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
        "true_negatives": tn,
        "avg_input_tokens": avg_tokens,
        "avg_latency_seconds": avg_latency,
        "avg_tool_calls_per_case": avg_tool_calls,
    }

print("Metrics calculator defined")

Metrics calculator defined


My views 
Claude might get "medium" risk category cases wrong. 
I expect high recall 
Medium category is highest to classify. 


In [70]:
results, errors = run_eval(test_cases, model="claude-haiku-4-5", use_tools=True)

Starting eval: 20 cases, model=claude-haiku-4-5
Use tools: True
--------------------------------------------------
Running case 1/20: C001... Claude is calling: check_sanctions({'entity_name': 'Northwind Trading LLC'})
Claude is calling: get_jurisdiction_risk({'country': 'Cyprus'})
Claude is calling: get_jurisdiction_risk({'country': 'Latvia'})
Claude is calling: get_jurisdiction_risk({'country': 'United Arab Emirates'})
CORRECT (predicted=high, truth=high, score=92, time=17.47s)
Running case 2/20: C004... Claude is calling: check_sanctions({'entity_name': 'Rashid Al-Farsi'})
Claude is calling: get_jurisdiction_risk({'country': 'Iran'})
CORRECT (predicted=high, truth=high, score=92, time=5.24s)
Running case 3/20: C005... Claude is calling: check_sanctions({'entity_name': 'Pacific Rim Holdings'})
Claude is calling: get_jurisdiction_risk({'country': 'Cayman Islands'})
Claude is calling: get_jurisdiction_risk({'country': 'Panama'})
Claude is calling: get_jurisdiction_risk({'country': 'Bri

In [71]:
metrics = calculate_metrics(results)
print("=== EVAL METRICS ===")
print(f"Accuracy:  {metrics['accuracy']}%")
print(f"Precision: {metrics['precision']}%")
print(f"Recall:    {metrics['recall']}%")
print(f"F1 Score:  {metrics['f1_score']}%")
print(f"\n=== CONFUSION MATRIX (High Risk vs Not High Risk) ===")
print(f"True Positives  (correctly flagged high):   {metrics['true_positives']}")
print(f"False Positives (wrongly flagged high):      {metrics['false_positives']}")
print(f"False Negatives (missed high risk):          {metrics['false_negatives']}")
print(f"True Negatives  (correctly cleared):         {metrics['true_negatives']}")
print(f"\n=== COST & PERFORMANCE ===")
print(f"Avg input tokens per case:  {metrics['avg_input_tokens']}")
print(f"Avg latency per case:       {metrics['avg_latency_seconds']}s")
print(f"Avg tool calls per case:    {metrics['avg_tool_calls_per_case']}")

=== EVAL METRICS ===
Accuracy:  100.0%
Precision: 100.0%
Recall:    100.0%
F1 Score:  100.0%

=== CONFUSION MATRIX (High Risk vs Not High Risk) ===
True Positives  (correctly flagged high):   5
False Positives (wrongly flagged high):      0
False Negatives (missed high risk):          0
True Negatives  (correctly cleared):         15

=== COST & PERFORMANCE ===
Avg input tokens per case:  1376
Avg latency per case:       6.18s
Avg tool calls per case:    2.7


In [72]:
eval_output = {
    "eval_timestamp": datetime.now().isoformat(),
    "model": "claude-haiku-4-5",
    "total_cases": 20,
    "metrics": metrics,
    "results": results,
    "errors": errors
}

with open("eval_results_haiku.json", "w") as f:
    json.dump(eval_output, f, indent=2)

print("Results saved to eval_results_haiku.json")

Results saved to eval_results_haiku.json


My expectations about comparing Haiku and Sonnet - Both will give the same answer and a higher cost model is not needed for the evalation. (Prove)

In [79]:
results_sonnet, errors_sonnet = run_eval(
    test_cases, 
    model="claude-sonnet-4-6", 
    use_tools=True
)

Starting eval: 20 cases, model=claude-sonnet-4-6
Use tools: True
--------------------------------------------------
Running case 1/20: C001... Claude is calling: check_sanctions({'entity_name': 'Northwind Trading LLC'})
Claude is calling: get_jurisdiction_risk({'country': 'Cyprus'})
Claude is calling: get_jurisdiction_risk({'country': 'UAE'})
Claude is calling: get_jurisdiction_risk({'country': 'Latvia'})
CORRECT (predicted=high, truth=high, score=92, time=15.06s)
Running case 2/20: C004... Claude is calling: check_sanctions({'entity_name': 'Rashid Al-Farsi'})
Claude is calling: get_jurisdiction_risk({'country': 'Iran'})
CORRECT (predicted=high, truth=high, score=94, time=11.78s)
Running case 3/20: C005... Claude is calling: check_sanctions({'entity_name': 'Pacific Rim Holdings'})
Claude is calling: get_jurisdiction_risk({'country': 'Cayman Islands'})
Claude is calling: get_jurisdiction_risk({'country': 'Panama'})
Claude is calling: get_jurisdiction_risk({'country': 'British Virgin Isl

In [77]:
# Diagnostic: see exactly what Sonnet returns for a failing case
case_text = json.dumps(test_cases[2], indent=2)  # C005

messages = [{"role": "user", "content": f"Evaluate this AML case using the available tools:\n\n{case_text}"}]

while True:
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1000,
        system=SYSTEM_PROMPT,
        tools=tools,
        messages=messages
    )
    
    print(f"stop_reason: {response.stop_reason}")
    print(f"content blocks: {len(response.content)}")
    for i, block in enumerate(response.content):
        print(f"\nBlock {i}: type={block.type}")
        if hasattr(block, 'text'):
            print(f"text repr: {repr(block.text)}")
    
    if response.stop_reason == "tool_use":
        messages.append({"role": "assistant", "content": response.content})
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                print(f"\nTool call: {block.name}({block.input})")
                if block.name == "check_sanctions":
                    result = check_sanctions(block.input["entity_name"])
                elif block.name == "get_jurisdiction_risk":
                    result = get_jurisdiction_risk(block.input["country"])
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps(result)
                })
        messages.append({"role": "user", "content": tool_results})
    
    elif response.stop_reason == "end_turn":
        print("\n=== FINAL RESPONSE REACHED ===")
        break

stop_reason: tool_use
content blocks: 5

Block 0: type=text
text repr: "I'll analyze this case by simultaneously checking sanctions status and jurisdiction risk for all relevant jurisdictions mentioned."

Block 1: type=tool_use

Block 2: type=tool_use

Block 3: type=tool_use

Block 4: type=tool_use

Tool call: check_sanctions({'entity_name': 'Pacific Rim Holdings'})

Tool call: get_jurisdiction_risk({'country': 'Cayman Islands'})

Tool call: get_jurisdiction_risk({'country': 'Panama'})

Tool call: get_jurisdiction_risk({'country': 'British Virgin Islands'})
stop_reason: end_turn
content blocks: 1

Block 0: type=text
text repr: 'All tool results are back. Every jurisdiction involved is rated high-risk — a critical compounding factor. Here is the full assessment:\n\n{"case_id":"C005","risk_score":96,"risk_category":"high","red_flags":["Unidentified ultimate beneficial owner (UBO) obscured behind three layers of holding companies — classic layering typology","Funds received exclusively fr

In [80]:
metrics_sonnet = calculate_metrics(results_sonnet)
print("=== SONNET EVAL METRICS ===")
print(f"Accuracy:  {metrics_sonnet['accuracy']}%")
print(f"Precision: {metrics_sonnet['precision']}%")
print(f"Recall:    {metrics_sonnet['recall']}%")
print(f"F1 Score:  {metrics_sonnet['f1_score']}%")
print(f"\n=== CONFUSION MATRIX ===")
print(f"True Positives:  {metrics_sonnet['true_positives']}")
print(f"False Positives: {metrics_sonnet['false_positives']}")
print(f"False Negatives: {metrics_sonnet['false_negatives']}")
print(f"True Negatives:  {metrics_sonnet['true_negatives']}")
print(f"\n=== COST & PERFORMANCE ===")
print(f"Avg input tokens:    {metrics_sonnet['avg_input_tokens']}")
print(f"Avg latency:         {metrics_sonnet['avg_latency_seconds']}s")
print(f"Avg tool calls:      {metrics_sonnet['avg_tool_calls_per_case']}")

print("\n\n=== HAIKU vs SONNET COMPARISON ===")
print(f"{'Metric':<25} {'Haiku':>10} {'Sonnet':>10}")
print("-" * 45)
print(f"{'Accuracy':<25} {metrics['accuracy']:>9}% {metrics_sonnet['accuracy']:>9}%")
print(f"{'Precision':<25} {metrics['precision']:>9}% {metrics_sonnet['precision']:>9}%")
print(f"{'Recall':<25} {metrics['recall']:>9}% {metrics_sonnet['recall']:>9}%")
print(f"{'F1 Score':<25} {metrics['f1_score']:>9}% {metrics_sonnet['f1_score']:>9}%")
print(f"{'Avg Input Tokens':<25} {metrics['avg_input_tokens']:>10} {metrics_sonnet['avg_input_tokens']:>10}")
print(f"{'Avg Latency (s)':<25} {metrics['avg_latency_seconds']:>10} {metrics_sonnet['avg_latency_seconds']:>10}")
print(f"{'Avg Tool Calls':<25} {metrics['avg_tool_calls_per_case']:>10} {metrics_sonnet['avg_tool_calls_per_case']:>10}")

=== SONNET EVAL METRICS ===
Accuracy:  90.0%
Precision: 71.4%
Recall:    100.0%
F1 Score:  83.3%

=== CONFUSION MATRIX ===
True Positives:  5
False Positives: 2
False Negatives: 0
True Negatives:  13

=== COST & PERFORMANCE ===
Avg input tokens:    1416
Avg latency:         14.14s
Avg tool calls:      2.9


=== HAIKU vs SONNET COMPARISON ===
Metric                         Haiku     Sonnet
---------------------------------------------
Accuracy                      100.0%      90.0%
Precision                     100.0%      71.4%
Recall                        100.0%     100.0%
F1 Score                      100.0%      83.3%
Avg Input Tokens                1376       1416
Avg Latency (s)                 6.18      14.14
Avg Tool Calls                   2.7        2.9


In [81]:
eval_output_sonnet = {
    "eval_timestamp": datetime.now().isoformat(),
    "model": "claude-sonnet-4-6",
    "total_cases": 20,
    "metrics": metrics_sonnet,
    "results": results_sonnet,
    "errors": errors_sonnet
}

with open("eval_results_sonnet.json", "w") as f:
    json.dump(eval_output_sonnet, f, indent=2)

print("Sonnet results saved")

Sonnet results saved
